# Imports

In [55]:
%reload_ext autoreload
%autoreload 2


from stnd.utility.data_utils import make_or_load_from_cache
from stnd.utility.utils import apply_random_seed
import sys
import os
import sklearn
import numpy as np
import pandas as pd

ROOT_PATH = os.path.join(os.path.dirname(os.path.abspath("")))
sys.path.insert(0, ROOT_PATH)
from experiments import (
    RANDOM_SEED,
    make_train_test_model_embeddings,
    make_cache_subpath,
    make_disagreement_scores_dict,
    make_fitted_weights
)
from utils import (
    lb_scenarios,
    dump_pickle,
    load_pickle,
    prepare_and_split_data
)
from plots import (
    MODEL_OUTPUTS_PATH,
    load_scores,
    safe_spearmanr
)
from selection import sample_items
from run_experiment import load_and_split_model_outputs
from acc import (
    compute_true_acc,
    compute_acc_knn
)
sys.path.pop(0);

# Evaluate new model

## Get ground truth

### Take from leaderboard snapshot

In [56]:
df = pd.read_csv(os.path.join(ROOT_PATH, "benchmark_csvs","open-llm-leaderboard.csv"))
df.head()


,T,Model,Average ⬆️,ARC,HellaSwag,MMLU,TruthfulQA,Winogrande,GSM8K,Type,...,Weight type,Precision,Merged,Hub License,#Params (B),Hub ❤️,Available on the hub,Model sha,Flagged,MoE
0,🟦,moreh/MoMo-70B-lora-1.8.6-DPO,77.29,70.14,86.03,77.40,69.00,84.37,76.80,RL-tuned,...,Original,bfloat16,False,mit,72.29,18.0,True,76389d5d825c3743cc70bc75b902bbfdad11beba,False,False
1,🟢,cloudyu/Yi-34Bx2-MoE-60B,76.72,71.08,85.23,77.47,66.19,84.85,75.51,pretrained,...,Original,bfloat16,False,cc-by-nc-4.0,60.81,42.0,True,483359d70b3fef480cdaeb6d722a18626d34f0ce,True,True
2,🟢,cloudyu/Mixtral_34Bx2_MoE_60B,76.66,71.33,85.25,77.34,66.59,84.85,74.60,pretrained,...,Original,bfloat16,False,cc-by-nc-4.0,60.81,83.0,True,f49d7cf0a7b99b15bc98b0ef4a681e7f0f4aa92c,True,True
3,🟢,cloudyu/Mixtral_34Bx2_MoE_60B,76.63,71.25,85.36,77.28,66.61,84.69,74.60,pretrained,...,Original,float16,False,cc-by-nc-4.0,60.81,83.0,True,f49d7cf0a7b99b15bc98b0ef4a681e7f0f4aa92c,True,True
4,🟦,moreh/MoMo-70B-lora-1.8.4-DPO,76.23,69.62,85.35,77.33,64.64,84.14,76.27,RL-tuned,...,Original,bfloat16,False,mit,72.29,10.0,True,a2c3a87dd53a87dc9fc622ce4ddbb05d3e9cf6a9,False,False


### Eval on the whole dataset to compute ground truth from scratch

In [ ]:
# eval

## DISCO-eval

In [ ]:
# get anchor points
# get predictor
# get outputs for anchor points, cache
# predict with predictor
# compare to ground truth

# Debug

In [2]:
bench = "mmlu_fields"
data, scenarios, set_of_rows, data_path = load_and_split_model_outputs(
    bench=bench,
    split='noniid',
    model_outputs_path=MODEL_OUTPUTS_PATH,
    text_to_vector=None,
    return_data_path=True,
    subsample_validation=False,
)

chosen_scenarios = list(scenarios.keys())
split_number = 0
rows_to_hide = set_of_rows[split_number]


40 425


In [47]:
print(data['data']['harness_arc_challenge_25']['correctness'].shape)

(1172, 425)


In [3]:
(
    scores_train,
    predictions_train,
    predictions_test,
    scores_test,
    balance_weights,
    scenarios_position,
    subscenarios_position,
) = prepare_and_split_data(
    chosen_scenarios,
    scenarios,
    data,
    rows_to_hide=rows_to_hide,
    n_source_models=None,
)

In [17]:
sampling_names = ["high-disagreement"]
disagreement_type = "pds"
disagreement_scores_dict = make_disagreement_scores_dict(
    config={
        "sampling_names": sampling_names,
        "predictions_train": predictions_train,
        "disagreement_type": disagreement_type,
    }
)

In [18]:

number_items = [100]
iterations = 5
sampling_name = sampling_names[0]
number_item = number_items[0]
seen_items_dic = {sampling_name: {}}
apply_random_seed(RANDOM_SEED)
samples = sample_items(
    number_item,
    iterations,
    sampling_name,
    chosen_scenarios,
    scenarios,
    subscenarios_position,
    responses_test=scores_test,
    scores_train=scores_train,
    predictions_train=predictions_train,
    scenarios_position=scenarios_position,
    A=None,
    B=None,
    balance_weights=balance_weights,
    disagreement_scores_dict=disagreement_scores_dict,
    skip_irt=True,
)
(
    _,
    seen_items_dic[sampling_name][number_item],
    _,
    _
) = samples

In [19]:
# load embeddings
# cache_path = ""
# cache = load_pickle(cache_path)
# scenario_name = "mmlu"
# split_number = 0
# emb_cache_path = (
#     make_cache_subpath(
#         cache, scenario_name, split_number, f"embeddings_path"
#     )
# )

pca = 256
apply_softmax_to_predictions = True
(
    train_models_embeddings,
    test_models_embeddings,
) = make_train_test_model_embeddings(
    emb_config={
        "sampling_names": sampling_names,
        "number_items": number_items,
        "iterations": iterations,
        "predictions_train": predictions_train,
        "seen_items_dic": seen_items_dic,
        "predictions_test": predictions_test,
        "seen_items_dic": seen_items_dic,
        "pca": pca,
        "apply_softmax": apply_softmax_to_predictions,
    }
)



# make_or_load_from_cache(
#     object_name="train_test_model_embeddings",
#     object_config={
#         "sampling_names": sampling_names,
#         "number_items": number_items,
#         "iterations": iterations,
#         "predictions_train": predictions_train,
#         "seen_items_dic": seen_items_dic,
#         "predictions_test": predictions_test,
#         "seen_items_dic": seen_items_dic,
#         "pca": pca,
#         "apply_softmax": apply_softmax_to_predictions,
#     },
#     make_func=make_train_test_model_embeddings,
#     cache_path=emb_cache_path,
# )

Computing embeddings: 100%|██████████| 1/1 [00:04<00:00,  4.76s/it]


In [21]:
train_model_indices = list(range(scores_train.shape[0]))
train_model_true_accs = compute_true_acc(
    scores_train,
    balance_weights,  # sample -> sample weight
    scenarios_position,  # scenario -> list of sample indices
    chosen_scenarios,
    train_model_indices,
    train_model_indices,  # they are not the global indices, but the contiguous indices of train models after removing test models
)

In [38]:

    # "RandomForestRegressor_100": {
    #     "class_path": "sklearn.ensemble.RandomForestRegressor",
    #     "params": {"n_estimators": 100}
    # },

fitted_model_type = "RandomForestRegressor_100"
chosen_fitting_methods = [
    (fitted_model_type, (sklearn.ensemble.RandomForestRegressor, {"n_estimators": 100}))
]
scenario = bench.split("_")[0]
fitted_weights = make_fitted_weights(
    config={
        "sampling_names": sampling_names,
        "number_items": number_items,
        "iterations": iterations,
        "train_models_embeddings": train_models_embeddings,
        "train_model_true_accs": train_model_true_accs,
        "scenario": scenario,
        "cache_path": "./cache_for_notebook",
        "chosen_fitting_methods": chosen_fitting_methods,
        "skip_iterations_when_fixed_sampling": False,
    }
)

# make_or_load_from_cache(
#     object_name="fitted_weights",
#     object_config={
#         "sampling_names": sampling_names,
#         "number_items": number_items,
#         "iterations": iterations,
#         "train_models_embeddings": train_models_embeddings,
#         "train_model_true_accs": train_model_true_accs,
#         "scenario": bench,
#         "cache_path": fitted_weights_cache_path,
#         "chosen_fitting_methods": chosen_fitting_methods,
#     },
#     make_func=make_fitted_weights,
#     cache_path=fitted_weights_cache_path,
# )

In [39]:
predictors_per_seed = fitted_weights[sampling_names[0]][100]
target_model_embeddings_per_seed = test_models_embeddings[sampling_names[0]][100]
predicted_accs = {}
predicted_accs_knn = {}
for seed in range(iterations):
    fitted_model = predictors_per_seed[seed][fitted_model_type]
    target_model_embeddings = target_model_embeddings_per_seed[seed]
    for target_model_idx in range(target_model_embeddings.shape[0]):
        test_model_embedding = target_model_embeddings[target_model_idx]
        fitted_acc = fitted_model.predict(
            test_model_embedding.numpy().reshape(1, -1)
        )[0]
        fitted_acc_knn = compute_acc_knn(
            test_model_embedding=test_model_embedding,
            train_model_embeddings=train_models_embeddings[
                sampling_name
            ][number_item][seed],
            scenario=scenario,
            train_model_true_accs=train_model_true_accs,
        )
        target_model = rows_to_hide[target_model_idx]
        if target_model not in predicted_accs:
            predicted_accs[target_model] = []
            predicted_accs_knn[target_model] = []
        predicted_accs[target_model].append(fitted_acc)
        predicted_accs_knn[target_model].append(fitted_acc_knn)

In [40]:
scores = load_scores(
    bench,
    split='noniid',
    scenarios_to_skip=[],
    ordered=True,
    filename_suffix="",
    num_it=5,
    data_path=None,
    filter_indices_by_results=False
)
gt_scores = scores[:, rows_to_hide]


In [41]:
maes_per_method = {}
rank_corrs_per_method = {}
predictions = {
    "fitted": predicted_accs,
    "knn": predicted_accs_knn
}
for method_name, accs in predictions.items():
    rank_corrs = []
    # pred_accs_as_np = np.stack(list(predicted_accs.values()), axis=0)
    pred_accs_as_np = np.stack(list(accs.values()), axis=0)
    maes = np.abs(
        pred_accs_as_np - gt_scores[0][:, None]
    )
    for i in range(pred_accs_as_np.shape[1]):
        rank_corrs.append(safe_spearmanr(
            pred_accs_as_np[:, i],
            gt_scores[0][:, None],
        ))
    maes_per_method[method_name] = maes
    rank_corrs_per_method[method_name] = np.array(rank_corrs)

In [42]:
for method_name, maes in maes_per_method.items():
    rank_corrs = rank_corrs_per_method[method_name]
    mae_str = f"{maes.mean(axis=1).mean()} +- {maes.std(axis=1).mean()}"

    rank_corrs_str = f"{rank_corrs.mean().mean()} +- {rank_corrs.std()}"
    print(f"{method_name}: {mae_str} | {rank_corrs_str}")

fitted: 0.012916896586372958 +- 0.0017361928302682007 | 0.9729831144465292 +- 0.001283499540449244
knn: 0.015829004036839338 +- 0.0 | 0.9477832773079722 +- 1.1102230246251565e-16
